In [ ]:
!apt-get update
!apt-get install -y chromium-chromedriver
!pip install selenium pandas openpyxl xlsxwriter urllib3

In [ ]:
# =============================
# IMPORTS
# =============================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys

import pandas as pd
from pandas import ExcelWriter
import requests
import urllib3
import os

# =============================
# CONFIG SELENIUM (COLAB)
# =============================
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

nami = webdriver.Chrome(options=options)

# =============================
# LOGIN
# =============================
from login import Login

login = "10631094776"
senha = "10631094776"

entrar = Login(nami, login, senha)

# =============================
# SCRIPT PRINCIPAL
# =============================
from meus_scripts import AlunosEmCurso

alunos_em_curso = AlunosEmCurso(nami, "1 TRIMESTRE")
links_turmas = alunos_em_curso.pega_links_notas()

# =============================
# CONFIG
# =============================
urllib3.disable_warnings()
os.makedirs("OUTPUT", exist_ok=True)

# =============================
# PROCESSAMENTO
# =============================
with ExcelWriter('OUTPUT/alunos_em_curso.xlsx') as w:
    
    for url in links_turmas:   
        
        cookies = {c['name']: c['value'] for c in nami.get_cookies()}

        data = requests.get(url, cookies=cookies, verify=False).json()

        sheet = data["classroom"]["name"]

        df = pd.DataFrame(data["group_students"])

        df_em_curso = df[df["status_humanize"] == "Em curso"][["number","name"]]

        df_em_curso.to_excel(
            w,
            sheet_name=sheet[:31],  # evita erro de nome grande
            freeze_panes=(1, 2),
            index=False
        )

# =============================
# FINALIZA
# =============================
nami.quit()

print("✅ Planilha gerada em OUTPUT/alunos_em_curso.xlsx")

In [ ]:
from google.colab import files
files.download('OUTPUT/alunos_em_curso.xlsx')